# Random Forest

Bagging + subamostragem de features aplicados a arvores de decisao.

Cada arvore ve um bootstrap diferente e, em cada split, escolhe entre
um subconjunto aleatorio de features. A predicao final e por voto
majoritario. O resultado tipico: menor variancia que uma arvore unica.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report,
)

In [ ]:
class Node:
    """No de uma arvore de decisao."""
    __slots__ = ("feature", "threshold", "left", "right", "value")

    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value  # preenchido apenas em folhas

    def is_leaf(self):
        return self.value is not None


class DecisionTreeClassifier:
    """
    Arvore de decisao CART para classificacao.

    Criterios suportados: 'gini' e 'entropy'.
    Split por feature continua usando limiares candidatos unicos.
    """

    def __init__(self, max_depth=None, min_samples_split=2, criterion="gini",
                 max_features=None, random_state=None):
        if criterion not in {"gini", "entropy"}:
            raise ValueError("criterion must be 'gini' or 'entropy'.")
        if min_samples_split < 2:
            raise ValueError("min_samples_split must be >= 2.")

        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.criterion = criterion
        self.max_features = max_features
        self.random_state = random_state

        self.root = None
        self.n_features_ = None
        self.classes_ = None
        self._rng = None

    def _impurity(self, y):
        if len(y) == 0:
            return 0.0
        _, counts = np.unique(y, return_counts=True)
        probs = counts / counts.sum()
        if self.criterion == "gini":
            return 1.0 - np.sum(probs ** 2)
        # entropia
        return -np.sum(probs * np.log2(probs + 1e-12))

    def _best_split(self, X, y, feature_indices):
        best_gain = -1.0
        best_feat, best_thr = None, None
        parent_impurity = self._impurity(y)
        n = len(y)

        for feat in feature_indices:
            values = X[:, feat]
            thresholds = np.unique(values)
            # usamos pontos medios entre valores ordenados como candidatos
            if len(thresholds) > 1:
                thresholds = (thresholds[:-1] + thresholds[1:]) / 2.0

            for thr in thresholds:
                left_mask = values <= thr
                right_mask = ~left_mask
                n_left = left_mask.sum()
                n_right = n - n_left

                if n_left == 0 or n_right == 0:
                    continue

                child_impurity = (
                    n_left / n * self._impurity(y[left_mask])
                    + n_right / n * self._impurity(y[right_mask])
                )
                gain = parent_impurity - child_impurity

                if gain > best_gain:
                    best_gain = gain
                    best_feat = feat
                    best_thr = thr

        return best_feat, best_thr, best_gain

    def _most_common_label(self, y):
        values, counts = np.unique(y, return_counts=True)
        return values[np.argmax(counts)]

    def _build(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        # condicoes de parada
        if (
            (self.max_depth is not None and depth >= self.max_depth)
            or n_samples < self.min_samples_split
            or n_labels == 1
        ):
            return Node(value=self._most_common_label(y))

        # feature subsampling (usado por Random Forest)
        if self.max_features is not None and self.max_features < n_features:
            feature_indices = self._rng.choice(
                n_features, size=self.max_features, replace=False
            )
        else:
            feature_indices = np.arange(n_features)

        feat, thr, gain = self._best_split(X, y, feature_indices)
        if feat is None or gain <= 0:
            return Node(value=self._most_common_label(y))

        left_mask = X[:, feat] <= thr
        left = self._build(X[left_mask], y[left_mask], depth + 1)
        right = self._build(X[~left_mask], y[~left_mask], depth + 1)
        return Node(feature=feat, threshold=thr, left=left, right=right)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        if X.ndim != 2:
            raise ValueError("X must be 2D.")

        self.n_features_ = X.shape[1]
        self.classes_ = np.unique(y)
        self._rng = np.random.default_rng(self.random_state)
        self.root = self._build(X, y)
        return self

    def _predict_sample(self, x, node):
        while not node.is_leaf():
            if x[node.feature] <= node.threshold:
                node = node.left
            else:
                node = node.right
        return node.value

    def predict(self, X):
        if self.root is None:
            raise ValueError("Call fit() before predict().")
        X = np.asarray(X, dtype=float)
        return np.array([self._predict_sample(x, self.root) for x in X])

    def score(self, X, y):
        return np.mean(self.predict(X) == np.asarray(y))

In [ ]:
class RandomForestClassifier:
    """
    Ensemble de arvores de decisao via bagging + subamostragem de features.

    Cada arvore e treinada em um bootstrap do dataset e, em cada split,
    considera apenas um subconjunto aleatorio de features (decorrelando
    os aprendizes fracos).
    """

    def __init__(self, n_estimators=100, max_depth=None, min_samples_split=2,
                 max_features="sqrt", criterion="gini", random_state=None):
        if n_estimators <= 0:
            raise ValueError("n_estimators must be positive.")

        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.criterion = criterion
        self.random_state = random_state

        self.trees_ = []
        self.classes_ = None

    def _resolve_max_features(self, n_features):
        if self.max_features == "sqrt":
            return max(1, int(np.sqrt(n_features)))
        if self.max_features == "log2":
            return max(1, int(np.log2(n_features)))
        if isinstance(self.max_features, int):
            return max(1, min(self.max_features, n_features))
        if self.max_features is None:
            return n_features
        raise ValueError("invalid max_features")

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        n_samples, n_features = X.shape

        self.classes_ = np.unique(y)
        max_feats = self._resolve_max_features(n_features)
        rng = np.random.default_rng(self.random_state)

        self.trees_ = []
        for i in range(self.n_estimators):
            # bootstrap
            idx = rng.integers(0, n_samples, size=n_samples)
            X_boot, y_boot = X[idx], y[idx]

            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                criterion=self.criterion,
                max_features=max_feats,
                random_state=int(rng.integers(0, 2**31 - 1)),
            )
            tree.fit(X_boot, y_boot)
            self.trees_.append(tree)

        return self

    def predict(self, X):
        if not self.trees_:
            raise ValueError("Call fit() before predict().")
        X = np.asarray(X, dtype=float)

        # coleta votos e escolhe por maioria
        tree_preds = np.array([tree.predict(X) for tree in self.trees_])
        preds = []
        for col in tree_preds.T:
            values, counts = np.unique(col, return_counts=True)
            preds.append(values[np.argmax(counts)])
        return np.array(preds)

    def score(self, X, y):
        return np.mean(self.predict(X) == np.asarray(y))

In [ ]:
data = datasets.load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42, stratify=data.target
)

rf = RandomForestClassifier(
    n_estimators=50, max_depth=8, max_features="sqrt", random_state=42
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=data.target_names).plot()
plt.title("Random Forest - Breast Cancer")
plt.show()